In [1]:
import numpy as np
import re

In [280]:
# Routines

def figures_routine(lines):
    latex = "".join(lines)
    ruta = re.search(r'\\includegraphics(?:\[[^\]]*\])?\{([^}]*)\}', latex)
    caption = re.search(r'\\caption\{(.*?)\}', latex, re.DOTALL)
    label= re.search(r'\\label\{([^}]*)\}', latex)
    if ruta and caption and label:
        xml = [f"""<figura id="{label.group(1)}">\n""",
        f"""    <ruta>{ruta.group(1)}</ruta>\n""",
        f"""    <caption>\n""",
        f"""        {caption.group(1)}\n""",
        f"""    </caption>\n""",
        f"""</figura>\n"""]
    return xml

def eqs_routine(lines):
    latex = "".join(lines)
    label = re.search(r'\\label\{([^}]*)\}', latex)
    eq = latex.split("\\be")[1].split("\\ee")[0].replace("\n","").split("\\label")[0]
    #eq = latex.split("\n")[1].split("\\label")[0]
    if label:
        xml=[f"""<ecuaciÃ³n id="{label.group(1)}">\n""",
             f"""    <![CDATA[ {eq} ]]>\n""", 
             """</ecuaciÃ³n>\n"""]
    else:
        xml=["""<ecuaciÃ³n>\n""",
             f"""    <![CDATA[ {eq} ]]>\n""", 
             """</ecuaciÃ³n>\n"""]
    return xml

def align_routine(lines):
    latex = "".join(lines)
    latex1 = latex.split("\\begin{alig")[1].split("\\end{align}")[0]
    label =  re.findall(r'\\label\{([^}]*)\}', latex1)
    #eqs=latex.split("\\begin{align}")[1].split("\\end{align}")[0].replace("\n","").replace("&","").replace("\\notag","").split("\\\\")
    eqs=latex.replace("\\begin{align}","").replace("\\end{align}","").replace("\n","").replace("\t","").replace("&","").replace("\\notag","").split("\\\\")
    if label:
        xml=["""<ecuaciÃ³n>\n"""]
        for i, eq_ in enumerate(eqs):
            eq = eq_.split("\\label")[0].replace(""""   ""","")
            try: 
                label = label[i] 
                xml.extend([f"""    <lÃ\xadnea id="{label}">""",f"""<![CDATA[ {eq} ]]>""","""</lÃ\xadnea>\n"""])
            except: xml.extend(["""    <lÃ\xadnea>""",f"""<![CDATA[ {eq} ]]>""","""</lÃ\xadnea>\n"""])
        xml.append("""</ecuaciÃ³n>\n""")
    else:
        xml=["""<ecuaciÃ³n>\n"""]
        for eq in eqs:
            xml.extend([f"""    <lÃ\xadnea>""",f"""<![CDATA[ {eq} ]]>""","""</lÃ\xadnea>\n"""])
        xml.append("""</ecuaciÃ³n>\n""")
    return xml


def references_routine(lines, reference_name):
    i=lines[0].find('{')
    j=lines[0].find(',')
    nick_name=lines[0][i+1:j]
    info={}
    for line in lines[1:]:
        if '%' in line:
            continue
        i = line.find('=')
        label=line[:i].replace(" ","").replace("\t","")
        if '}' in label:
            continue
        j = line.find('{')
        k=j+1
        for l in range(1,len(line)):
            if line[-l] == '}':
                k=l
                break
        content = line[j+1:len(line)-k]
        info[label]=content.replace("&","&amp;").replace("<","&lt;").replace(">","&lt;")
    xml=["<"+reference_name+f""" id="{nick_name}">\n"""]
    
    label_order = ['author','year','title','journal','volume','number','pages']
    for e in label_order:
        if e in info:
            xml.append("\t<"+e+"> "+info[e]+" </"+e+">\n")
            del info[e]
    for e in info:
        if e=='': continue
        xml.append("\t<"+e+"> "+info[e]+" </"+e+">\n")
    xml.append("</"+reference_name+">\n\n")

    return xml

# Secciones

In [224]:
with open('ch1.txt','r') as f:
    lines = f.readlines()

In [52]:
xml_lines=[]; i=0; enable_paragraph = False

while i<len(lines):
    line = lines[i]
    if ("<" in line) or (">" in line): print(f"WARNING of '<' or '>' in line {i}")
    if "table" in line: print(f"Notification of \\table in line {i}")
    if "&" in line: print(f"WARNING of '&' in line {i}")

    if "\\begin{figure}" in line:
        for j, jline in enumerate(lines[i:]):
            if "\\end{figure}" in jline:
                xml = figures_routine(lines[i:i+j+1])
                xml_lines.extend(xml)
                i += j
                break
    elif "\\begin{subequations}" in line:
        for j, jline in enumerate(lines[i:]):
            if "\\end{subequations}" in jline:
                xml = align_routine(lines[i+1:i+j])
                xml_lines.extend(xml)
                i += j
                break
    elif "\\begin{align}" in line:
        for j, jline in enumerate(lines[i:]):
            if "\\end{align}" in jline:
                xml = align_routine(lines[i:i+j+1])
                xml_lines.extend(xml)
                i += j
                break
    elif ("\\be\n" in line) or ("\\be \n" in line) or ("\\be\t\n" in line):
        for j, jline in enumerate(lines[i:]):
            if "\\ee" in jline:
                xml = eqs_routine(lines[i:i+j+1])
                xml_lines.extend(xml)
                i += j
                break
    elif line == "\n":
        pass
    else:
        line = line.replace("&","&amp;").replace("<","&lt;").replace(">","&lt;")
        if enable_paragraph:
            xml=["<pÃ¡rrafo>",line, "</pÃ¡rrafo>\n"]
            xml_lines.extend(xml)
        else: xml_lines.append(line)
    i+=1 

WARNING of '<' or '>' in line 6
WARNING of '<' or '>' in line 12


In [53]:
#Add extra tabs
num = 2
for i in range(len(xml_lines)):
    xml_lines[i]="\t"*num + xml_lines[i]

In [125]:
with open('output.txt', 'w') as f:
    f.writelines(xml_lines)

In [123]:
xml_lines = []
for line in lines:
    line = line.replace("\n","")
    xml_lines.append("<comando> "+line+" </comando>\n")

# Referencias 

In [292]:
with open('ch1.txt','r') as f:
    lines = f.readlines()

In [293]:
xml_lines=[]; i=0; enable_paragraph = False

while i<len(lines):
    line = lines[i]
    if ("<" in line) or (">" in line): print(f"WARNING of '<' or '>' in line {i}")
    if "table" in line: print(f"Notification of \\table in line {i}")
    if "&" in line: print(f"WARNING of '&' in line {i}")

    if "%" in line:
        i+=1
        continue
    
    if "@" in line:
        start = line.find("@")
        end = line[start:].find("{")
        reference_name = line[start+1:end]
        for j, jline in enumerate(lines[i+1:]):
            if "@" in jline:
                xml = references_routine(lines[i:i+j],reference_name)
                xml_lines.extend(xml)
                #print(reference_name)
                #print(lines[i:i+j])
                i += j-1
                break
            elif j==len(lines[i+1:])-1:
                xml = references_routine(lines[i:i+j+1],reference_name)
                xml_lines.extend(xml)
                #print(reference_name)
                #print(lines[i:i+j+1])
                i += j-1
                break
    i+=1 

In [294]:
#Add extra tabs
num = 2
for i in range(len(xml_lines)):
    xml_lines[i]="\t"*num + xml_lines[i]

In [295]:
with open('output.txt', 'w') as f:
    f.writelines(xml_lines)